### Spider dataset analysis

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
from statistics import median
from textwrap import indent
from typing import get_args

from text2sql.data import build_samples
from text2sql.prompt import build_completion, build_prompt
from text2sql.spider import Split, load_examples, load_schemas, query_error

spider_dir = Path("../data/spider_data")
splits = get_args(Split)

examples = {s: load_examples(spider_dir, s) for s in splits}
dbs = {s: {ex["db_id"] for ex in examples[s]} for s in splits}
schemas = load_schemas(spider_dir, test=True)  # test_tables.json covers all 206 databases

Prompt and completion messages the model gets for the first dev question.

In [ ]:
sample = build_samples(spider_dir, "dev", clean=False)[0]

for message in build_prompt(sample) + build_completion(sample):
    print(f"{message['role']}:")
    print(indent(message["content"], "    "), end="\n\n")

Number of questions and distinct databases per split.

In [ ]:
print(f"{'split':13s} {'questions':>9s} {'databases':>9s}")
for s in splits:
    print(f"{s:13s} {len(examples[s]):9d} {len(dbs[s]):9d}")

Number of databases that appear in both splits of each pair.

In [ ]:
print(f"{'split':13s} {'split':13s} {'shared databases':>16s}")
for a, b in combinations(splits, 2):
    print(f"{a:13s} {b:13s} {len(dbs[a] & dbs[b]):16d}")

Median and max number of tables and columns per database, over the databases each split uses.

In [ ]:
print(f"{'split':13s} {'tables':>15s} {'columns':>15s}")
print(f"{'':13s} {'median':>7s} {'max':>7s} {'median':>7s} {'max':>7s}")
for s in splits:
    tables = [len(schemas[db]["table_names_original"]) for db in dbs[s]]
    columns = [len(schemas[db]["column_names_original"]) - 1 for db in dbs[s]]  # without "*"
    print(
        f"{s:13s} {median(tables):7.1f} {max(tables):7d} {median(columns):7.1f} {max(columns):7d}"
    )

Share of gold queries that contain each SQL keyword, per split.

In [ ]:
keywords = [
    "JOIN",
    "WHERE",
    "GROUP BY",
    "HAVING",
    "ORDER BY",
    "LIMIT",
    "DISTINCT",
    "INTERSECT",
    "UNION",
    "EXCEPT",
]
queries = {s: [" ".join(ex["query"].upper().split()) for ex in examples[s]] for s in splits}

print(" " * 10 + "".join(f"{s:>14s}" for s in splits))
for kw in keywords:
    print(
        f"{kw:10s}"
        + "".join(f"{sum(kw in q for q in queries[s]) / len(queries[s]):14.0%}" for s in splits)
    )

Number of repeated questions on the same database, per split.

In [ ]:
print(f"{'split':13s} {'duplicates':>10s}")
for s in splits:
    counts = Counter((ex["db_id"], ex["question"]) for ex in examples[s])
    print(f"{s:13s} {sum(n - 1 for n in counts.values()):10d}")

Gold queries that raise an error when run on their database.

In [ ]:
print(f"{'split':13s} {'index':>5s}  {'database':24s} error")
for s in splits:
    for i, ex in enumerate(examples[s]):
        error = query_error(spider_dir, ex["db_id"], ex["query"], test=s == "test")
        if error:
            print(f"{s:13s} {i:5d}  {ex['db_id']:24s} {error}")